# C++ 第 6 章：继承多态与综合实践

> **开始前：选择 C++17 内核。** 本章不能使用 C17 内核。在线实验使用固定数据或 `std::istringstream`，键盘输入 `std::cin` / `std::getline(std::cin, ...)` 放在本地完整程序中练习。

**先修**：C 部分的变量、控制语句、数组、指针和函数；按 C++ 章节顺序学习。

**本章目标**：区分组合与继承；通过基类引用使用虚函数；独立完成成绩报告。

**学习方法**：先预测，再运行、修改并解释。块内局部变量可重复运行；修改函数、类或模板后，使用 Kernel → Restart Kernel and Run All Cells。各章互不依赖内核状态。

**保存作业**：见 [课程入口](https://github.com/luckymark/c_course#保存作业与继续学习)。

## 1. 先问“有一个”还是“是一种”

学生有成绩，适合组合；不同评分规则都遵守同一个接口，才考虑继承。public 继承表达可替换的接口关系，不是单纯为了复用几个字段。虚函数在通过基类指针或引用调用时进行动态分派；纯虚函数 =0 使基类成为抽象类。override 让编译器检查是否真的覆盖基类方法。

In [ ]:
class PassRule {
public:
    virtual ~PassRule() = default;
    virtual bool passed(int score) const = 0;
};
class StandardRule final : public PassRule {
public:
    bool passed(int score) const override { return score >= 60; }
};
class StrictRule final : public PassRule {
public:
    bool passed(int score) const override { return score >= 80; }
};

In [ ]:
#include <iostream>
#include <memory>
#include <vector>
{
    std::vector<std::unique_ptr<PassRule>> rules;
    rules.push_back(std::make_unique<StandardRule>());
    rules.push_back(std::make_unique<StrictRule>());
    for (const auto& rule : rules) { std::cout << rule->passed(70) << '\n'; }
}

基类析构函数为 virtual，才能通过基类所有者正确析构派生对象。这里 unique_ptr 自动释放；不写 delete。把派生对象按值复制给可实例化基类会发生对象切片，丢失派生部分；多态接口用引用或指针传递。本例基类为抽象类，因此根本不能创建其值对象。

若区别只是一条分数线，用一个带 threshold 字段的类型通常更简单；本例用于理解动态分派，不要求综合项目人为加入继承。

## 小练习：用只读对象表示可配置的分数线

只填写 BEGIN ANSWER 与 END ANSWER 之间的代码。先预测，再运行；初始 FAIL 表示尚未作答。完成后应全部 PASS。

In [ ]:
#include <iostream>
{
    class Threshold {
    public:
        explicit Threshold(int minimum) : minimum_{minimum} {}
        bool passed(int score) const {
            /* BEGIN ANSWER */
            (void)score;
            (void)minimum_; // 占位：完成答案时替换本行与下一行。
            return false;
            /* END ANSWER */
        }
    private:
        int minimum_;
    };
    const Threshold rule{80};
    std::cout << "79: " << (!rule.passed(79) ? "PASS" : "FAIL") << '\n';
    std::cout << "80: " << (rule.passed(80) ? "PASS" : "FAIL") << '\n';
}

<details><summary>提示</summary>

先检查空数据和边界，再处理一般情况；只计算题目需要的返回值，不改变输入。

</details>

<details><summary>参考答案：完成后再展开</summary>

```cpp
return score >= minimum_;
```

</details>

## 2. 综合任务：从空白文件生成成绩报告

**输入约定**：第一行人数0..100，之后每行一个不含空格的 ASCII 姓名和0..100整数成绩；姓名不得重复。拒绝缺失记录、尾随垃圾字符、越界及重复姓名。输出按姓名排序的成绩，再输出人数、及格人数与平均分；空班级平均分显示 N/A。

先画数据流：输入文本 → 逐行验证 → map → 只读统计函数 → 输出。用 enum class 表示读取状态，避免魔法整数；用 optional 表示空班级无平均分。标准容器管理资源，不需要裸 new/delete。

**独立完成**：先不展开参考答案。至少记录正常、空班级、0和100、重复姓名、缺失成绩、非法后缀六类测试的实际结果。再给统计函数增加最高分，解释空班级如何表示。

<details><summary>完整参考程序：完成独立作答后再展开</summary>

## 本地实践：完整 C++17 程序

把代码保存为 `lesson.cpp`，在终端执行 `c++ -std=c++17 -Wall -Wextra -Wpedantic lesson.cpp -o lesson`，编译成功后运行 `./lesson`。Windows GCC 使用 `g++` 和 `-o lesson.exe`，PowerShell 运行 `.\lesson.exe`。完整程序不要直接放入在线单元或手动调用 main。

```cpp
#include <iostream>
#include <iomanip>
#include <map>
#include <optional>
#include <sstream>
#include <string>
enum class InputStatus { ok, invalid };
struct ParsedStudent {
    InputStatus status;
    std::string name;
    int score;
};
ParsedStudent parse_student(const std::string& line) {
    std::istringstream input{line};
    std::string name, extra;
    int score{};
    if (!(input >> name >> score) || score < 0 || score > 100 || (input >> extra)) {
        return {InputStatus::invalid, "", 0};
    }
    return {InputStatus::ok, name, score};
}
struct Summary {
    std::size_t count;
    int passed;
    std::optional<double> average;
};
Summary summarize(const std::map<std::string, int>& scores) {
    int total{}, passed{};
    for (const auto& entry : scores) {
        total += entry.second;
        if (entry.second >= 60) { ++passed; }
    }
    if (scores.empty()) { return {0, 0, std::nullopt}; }
    return {scores.size(), passed, static_cast<double>(total) / scores.size()};
}
int main() {
    std::string line, extra;
    if (!std::getline(std::cin, line)) { std::cout << "invalid\n"; return 1; }
    std::istringstream header{line};
    int count{};
    if (!(header >> count) || count < 0 || count > 100 || (header >> extra)) {
        std::cout << "invalid\n"; return 1;
    }
    std::map<std::string, int> scores;
    for (int i = 0; i < count; ++i) {
        if (!std::getline(std::cin, line)) { std::cout << "invalid\n"; return 1; }
        const auto student = parse_student(line);
        if (student.status != InputStatus::ok || !scores.emplace(student.name, student.score).second) {
            std::cout << "invalid\n"; return 1;
        }
    }
    if (std::cin >> extra) { std::cout << "invalid\n"; return 1; }
    const auto report = summarize(scores);
    for (const auto& [name, score] : scores) { std::cout << name << ':' << score << '\n'; }
    std::cout << "count=" << report.count << " passed=" << report.passed << " average=";
    if (report.average) { std::cout << std::fixed << std::setprecision(2) << *report.average; }
    else { std::cout << "N/A"; }
    std::cout << '\n';
}
```

验收输入与输出（`\n` 表示换行）：

- 输入 `"3\nBob 90\nAlice 85\nCarol 58\n"` → 输出 `"Alice:85\nBob:90\nCarol:58\ncount=3 passed=2 average=77.67\n"`，退出码 0。
- 输入 `"0\n"` → 输出 `"count=0 passed=0 average=N/A\n"`，退出码 0。
- 输入 `"2\nA 0\nB 100\n"` → 输出 `"A:0\nB:100\ncount=2 passed=1 average=50.00\n"`，退出码 0。
- 输入 `"2\nA 60\nA 80\n"` → 输出 `"invalid\n"`，退出码 1。
- 输入 `"2\nA 60\n"` → 输出 `"invalid\n"`，退出码 1。
- 输入 `"1\nA 101\n"` → 输出 `"invalid\n"`，退出码 1。
- 输入 `"1\nA 60x\n"` → 输出 `"invalid\n"`，退出码 1。
- 输入 `"1\nA 60 extra\n"` → 输出 `"invalid\n"`，退出码 1。
- 输入 `"-1\n"` → 输出 `"invalid\n"`，退出码 1。
- 输入 `"101\n"` → 输出 `"invalid\n"`，退出码 1。
- 输入 `"one\n"` → 输出 `"invalid\n"`，退出码 1。
- 输入 `"1\nA 999999999999999999999\n"` → 输出 `"invalid\n"`，退出码 1。
- 输入 `"0\nextra\n"` → 输出 `"invalid\n"`，退出码 1。
- 输入 `""` → 输出 `"invalid\n"`，退出码 1。

该程序在指定记录后还检查多余输入。交互运行需发送 EOF：macOS/Linux 在新行按 Ctrl+D；Windows终端通常为 Ctrl+Z 后回车。也可在 macOS/Linux 或 Windows命令提示符用输入重定向读取样例文件。

</details>

## 分层练习与自查

1. **基础**：说明同样调用 passed(70) 为何得到不同结果。
2. **改错**：指出缺少 virtual 析构函数和按值传递基类对象的问题。
3. **综合**：完成上面的空白文件任务，再与折叠参考程序比较。
4. **选做**：拆成头文件、统计实现、main 三个文件，分别编译后链接；或使用 ifstream/ofstream 读写文件并检查打开失败。
5. **自查**：说明容器、const引用、optional、RAII各自解决什么问题，以及为什么本项目无需继承。